# **Machine Learning on Big Data (CN7030) CRWK 23-24 Term B [60% weighting]**
# **Group ID: 3**
1.   Jorgo Luka - U1610157
---

If you want to add comments on your group work, please write it here for us:

# **Initiate and Configure Spark**

In [ ]:
!pip3 install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=ccc03a7edd3034f028c4e01e4e1853c456a20abdf28b68c13f2e83795b658bb6
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("IDS Intrusion Detection") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

spark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Dataset Size: 358.22MB

# Load the compressed file as a text file
df=spark.read.csv("/content/drive/MyDrive/Files/UniversityServerLogsDataset.csv", header = True)

# Display the DataFrame
df.show(20)

# more info
print(df.count())
print(df.rdd.getNumPartitions())

---
# **Task 1 - Data Loading and Preprocessing (15 marks)**
---

In [ ]:
from pyspark.sql.functions import col, isnan, when, count
# Count NaN values in each column
nan_counts = df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns])

# Show the counts
nan_counts.show()

In [ ]:
#from pyspark.sql.functions import col, when, count

# Count infinity values in each column
inf_counts = df.select([count(when((col(c) == float('inf')) | (col(c) == float('-inf')), c)).alias(c) for c in df.columns])

# Show the counts
inf_counts.show()

In [ ]:
filtered_df = df.filter((col('Flow Duration') <= 0) & (col('Tot Fwd Pkts') <= 0) & (col('Tot Bwd Pkts') <= 0))

In [ ]:
# To go ahead with the full size, repartiton it to 6 chunks.

# for the 10% of data:
df = df.repartition(4)
df.rdd.getNumPartitions()

In [ ]:
df = df.dropna()

In [ ]:
df.select("Label").distinct().show(30)

In [ ]:
from pyspark.sql.functions import when

df = df.withColumn("label", when(df["Label"] != 'Benign', 1).otherwise(0))

df.select("label").distinct().show()

In [ ]:
df.groupBy("label").count().show()

# ~20% normal traffic
# ~80% attack traffic


# dealing with imbalanced labels:
 # Resampling
 # Weighted Loss
 # Data Augmentation: SMOTE method
 # Ensemble methods
 # Evaluation metrics: precision, recall, F1 score, AUC-ROC

In [ ]:
#Predicts binary outcomes
from pyspark.ml.feature import StringIndexer

columns_to_index = ["Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"]

for column in columns_to_index:
  indexer = StringIndexer(inputCol = column, outputCol = column + "_indexed")

#Converts categorical variables into numerical indices
  df = indexer.fit(df).transform(df)

df.show(5)

In [ ]:
# We will calculate the average and standard deviation for Flow Duration by class
from pyspark.sql.functions import avg, stddev

# avg & std for class 0
class_0_stats = df.filter(df['label'] == 0).select(avg('Flow Duration',).alias("avg_Flow Duration_0"), stddev('Flow Duration').alias("stddev_Flow Duration_0")).first()

# avg & std for class 1
class_1_stats = df.filter(df['label'] == 1).select(avg('Flow Duration').alias("avg_Flow Duration_1"),
                                                   stddev('Flow Duration').alias("stddev_Flow Duration_1")).first()


print("Class 0 distribution: ")
print("AVG Flow Duration: ", class_0_stats["avg_Flow Duration_0"])
print("STD Flow Duration: ", class_0_stats["stddev_Flow Duration_0"])

print("Class 1 distribution: ")
print("AVG Flow Duration: ", class_1_stats["avg_Flow Duration_1"])
print("STD Flow Duration: ", class_1_stats["stddev_Flow Duration_1"])

In [ ]:
df.printSchema()

In [ ]:
# Converts all selected columns to double type

from pyspark.sql.functions import col

for column in df.columns:
  df = df.withColumn(column, col(column).cast("double"))

In [ ]:
df.printSchema()

In [ ]:
#Preserve and include invalid records in the output dataset.

assembler = VectorAssembler(inputCols = ["Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"],

                            outputCol = "features", handleInvalid = "keep")

data = assembler.transform(df)

data = data.select('features', 'label')
data.show(5)

In [ ]:
#Truncate - To shorten or reduce the length of a number

data.show(5, truncate = False)

# Sparse Vector - with mostly zero or small values.

In [ ]:
# Convert sparse vector to dense list.

selected_data = data.select('features').limit(2).collect()

for row in selected_data:
  dense_vector = row[0].toArray()
  print(dense_vector)

In [ ]:
# Apply StandardScaler algorithm to the 'features' column of the data, transforming it into a new column called'scaledFeatures', and then display the first 3 rows of the data

scaler = StandardScaler(inputCol = 'features', outputCol = 'scaledFeatures')

scaler_model = scaler.fit(data)
data = scaler_model.transform(data)

data = data.select("scaledFeatures", "label")
data.show(3, truncate = False)

---
# **Task 2 - Model Selection and Implementation (25 marks)**
---


In [ ]:
#Split data
#Seed - A fixed number used to reproduce identical random data splits.

train_data, test_data = data. randomSplit([0.8, 0.2], seed = 1234)

In [ ]:
lr = LogisticRegression(featuresCol = "scaledFeatures", labelCol = 'label',
                        threshold = 0.5, regParam = 0.01)

lr_model = lr.fit(train_data)

lr_predictions_train = lr_model.transform(train_data)
lr_predictions_test = lr_model.transform(test_data)

In [ ]:
# Displaying predicted labels and actual labels for test data.
lr_predictions_test.select("label", "prediction").show(20)

---
# **Task 3 - Model Parameter Tuning (20 marks)**
---


In [ ]:
# A technique to optimize model hyperparameters through cross-validation and evaluation.

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder,CrossValidator

# Initialize Logistic Regression model
lr = LogisticRegression(featuresCol='scaledFeatures', labelCol='label')

# Define a grid of hyperparameters to search over
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.05, 0.7, 1]) \
    .build()

# Define an evaluator for binary classification
evaluator = BinaryClassificationEvaluator(metricName='areaUnderPR')

# Create a CrossValidator to tune the model
crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=param_grid,
                          evaluator=evaluator,
                          numFolds=5)

# Fit the CrossValidator to find the best model
cv_model = crossval.fit(train_data)

# Get the best Logistic Regression model from cross-validation
best_lr_model = cv_model.bestModel

# Make predictions on test data using the best model
predictions = best_lr_model.transform(test_data)

# Evaluate the model using the evaluator
auc = evaluator.evaluate(predictions)
print("Area Under ROC (AUC):", auc)




---
# **Task 4 - Model Evaluation and Accuracy Calculation (20 marks)**
---

In [ ]:
confusion_matrix = lr_predictions_test.groupBy("label", "prediction").count()
confusion_matrix.show()

In [ ]:
cm_pandas = confusion_matrix.toPandas()
cm_pandas.pivot(index = 'label', columns = 'prediction', values = 'count')

In [ ]:
tp = lr_predictions_test[(lr_predictions_test.label == 1) & (lr_predictions_test.prediction == 1)].count()
fp = lr_predictions_test[(lr_predictions_test.label == 0) & (lr_predictions_test.prediction == 1)].count()
fn = lr_predictions_test[(lr_predictions_test.label == 1) & (lr_predictions_test.prediction == 0)].count()
tn = lr_predictions_test[(lr_predictions_test.label == 0) & (lr_predictions_test.prediction == 0)].count()

---
# **Task 5 - Results Visualization or Printing (5 marks)**
---

In [ ]:
accuracy = (tp + tn) / (tp + fp + fn + tn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)

print('accuracy: ', round(accuracy, 4) * 100)
print('precision: ', round(precision, 4) * 100)
print('recall: ', round(recall, 4) * 100)
print('f1 score: ', round(f1, 4) * 100)

In [ ]:
import matplotlib.pyplot as plt

# Define the metrics and corresponding values
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values = [round(accuracy * 100, 4), round(precision * 100, 4), round(recall * 100, 4), round(f1 * 100, 4)]

colors = ['blue', 'green', 'orange','red']

# Create a bar chart
for i, (metric, value) in enumerate(zip(metrics, values)):
    plt.bar(metric, value, color=colors[i])

plt.xlabel('Metrics')
plt.ylabel('Values (%)')
plt.title('Model Performance')
plt.show()

---
# **Task 6 - LSEP Considerations (5 marks)**
---

Data Privacy and Security Considerations for Network Traffic Analysis

Data Privacy Concerns: The dataset contains personal data, including IP addresses, usernames, and passwords, which should be anonymized to protect individuals' privacy and prevent unauthorized access. This can be achieved through pseudonymization or removal of personal data.

Security Concerns: The dataset contains sensitive information about potential security breaches, which must be handled securely to prevent unauthorized access. This includes ensuring that the data is stored in a secure environment and that access is restricted to authorized personnel.

Stakeholder Engagement: The analysis of this dataset may have implications for various stakeholders, including network administrators, security professionals, and end-users. It is essential to engage with these stakeholders throughout the analysis process to ensure that their concerns and needs are addressed.

Cultural Sensitivity: The dataset may contain information about different cultures and communities, which could be sensitive and require cultural sensitivity. Therefore, it’s important to conduct the analysis in a culturally sensitive manner and avoid cultural biases.

Mitigation Strategies

Anonymization: Personal data will be anonymized to protect individuals' privacy and prevent unauthorized access.

Secure Handling: The data will be handled securely to prevent unauthorized access and maintain confidentiality.

Stakeholder Engagement: Stakeholders will be engaged throughout the analysis process to ensure that their concerns and needs are addressed.

Cultural Sensitivity: The analysis will be conducted in a culturally sensitive manner and cultural biases will be avoided.

Transparency: This will be ensured throughout the analysis process, including the methods used, data sources, and results.

---

# **Task 7 - Convert ipynb to HTML for Turnitin submission [5 marks]**

---



In [ ]:
# install nbconvert (if facing the conversion error)
!pip3 install nbconvert

In [ ]:
# convert ipynb to html and submit this HTML file
!jupyter nbconvert --to html Your_Group_ID_CRWK_CN7030.ipynb